# Day 32：高级RAG技术(上)：Query改写、Multi-Query、HyDE与上下文压缩（Jupyter Notebook 实战手册）

> **所属阶段**：第 3 阶段 · 第三阶段：LangChain 与企业级 RAG 开发 ｜ **第 5 周**：第5周：企业级 RAG 全链路、高级检索优化与评估  
> **GitLab 工单**：`STARMIND-032` ｜ **开发分支**：`feat/day32-advanced-rag-query-transform`  
> **配套 HTML5 课件**：`courseware/day32/index.html` ｜ **独立版本快照**：`versions/day32/`

---

## 🎙️ 老陈开场旁白

早啊！昨天看了员工内测日志，大家是不是觉得用户太不按套路出牌了？其实这才是真实的企业世界——**用户的提问（Query）和知识库里的文档（Document）天然存在「表达鸿沟」**！文档写的是「一线城市住宿标准」「生产发布窗口」「零信任 VPN 动态令牌」，员工问的是「魔都住店」「礼拜五推线上」「居家连不上内网」。今天老陈教大家高级 RAG 的前半套绝活——**检索前优化（Pre-Retrieval）与检索后压缩（Post-Retrieval Compression）**：
1. **Query Rewriting**：把口语、黑话、错别字翻译成规范检索词；
2. **Multi-Query**：一个问题从 3 个不同视角改写并行检索再融合去重；
3. **HyDE（假设性文档嵌入）**：先让大模型瞎编一段“看起来像制度原文的假设答案”，拿这段假设答案的向量去搜真实文档（文档搜文档，相似度暴涨！）；
4. **Contextual Compression**：把检索回来 500 字切片里无关的废话裁掉，只留最精准的句子喂给大模型省钱提效！

### 承上启下
- **昨日基线**：昨天（Day 31）我们用消融实验摸清了 Naive RAG 的最佳参数。但产品经理把第一批员工内测日志发过来，大家一看傻眼了：员工根本不按制度原文提问！他们问的是「去魔都出差住店能报多少？」「礼拜五下班前能不能推线上？」「电脑连不上公司内网咋整？」。这种口语/简称提问，单路 Naive 检索召回率直接掉了一大截。
- **明日预告**：明天（Day 33）我们将进入高级 RAG 技术（下）：攻克专有名词和精确编号检索难题，亲手实现 BM25 + 向量混合检索、RRF 倒数排名融合、bge-reranker 重排序与父文档检索器！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day32'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-032）

内测期间统计发现，约 35% 的员工提问存在地域别称（如魔都/帝都）、口语化表达（推线上/住店/请病假条）或复合多意图提问，导致 Naive RAG 单次直接检索漏召回。

**今日迭代目标：**
- 实现 `QueryRewriter`（术语规范化与口语改写）、`MultiQueryRetriever`（多视角扩展 + 去重合并）和 `HyDERetriever`（假设性文档嵌入检索）。
- 实现 `ContextualCompressor`（句子级相关性提纯，过滤低相关噪音句并统计压缩率）。
- 在口语化/黑话测试集上运行 Baseline vs QueryRewrite vs Multi-Query vs HyDE 召回率对比实验。

**验收标准：**
- 针对「魔都住店」「礼拜五推线上」等口语难题，QueryRewrite / Multi-Query / HyDE 的召回率显著高于 Naive 基线。
- 上下文压缩器在保留核心答案句子的前提下，平均压缩掉至少 35% 的冗余字符。
- 所有 pytest 单元测试与对比实验脚本 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. HyDE 虽好，但有一个致命适用边界！

HyDE 依赖大模型根据问题先「编」出一段符合领域风格的假设文档。对于通用概念（如报销流程、代码发布、休假规定），大模型非常擅长生成贴近原文的假设陈述；但如果你问的是公司内部特有代号（比如「星辰内部代号『天枢项目』的负责人是谁？」），大模型根本不知道天枢是什么，强行用 HyDE 瞎编反而会把向量带偏！

> **⚠️ 老陈避坑**：踩坑提醒：遇到专有项目代号、工单号、人名时不要用 HyDE，应该用明天（Day 33）的 **BM25 关键词 + 向量混合检索**！

### 2. 2. 生产环境 LangChain 对应组件写法对照

在 LangChain 生态中，Multi-Query 和上下文压缩都有现成封装：
```python
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

mq_retriever = MultiQueryRetriever.from_llm(retriever=vectorstore.as_retriever(), llm=fast_llm)
compressor = LLMChainExtractor.from_llm(fast_llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=mq_retriever
)
```

> **⚠️ 老陈避坑**：踩坑提醒：Multi-Query 和 LLM 上下文压缩都会增加额外的 LLM 调用！生产环境务必给它们配「7B/轻量小模型 + 并发执行」，不要拿昂贵的旗舰模型去干改写和压缩的脏活，否则 P95 延迟直接飙到 8 秒以上。


## 💻 实操步骤 1：步骤一：实现 QueryRewriter、MultiQueryRetriever 与 HyDERetriever

- **目标文件**：`versions/day32/src/day32_query_transform.py`
- **老陈代码旁白**：咱们一口气把检索前优化三板斧全部手写实现：针对星辰内部黑话与口语的 `QueryRewriter`、多视角去重聚合的 `MultiQueryRetriever`，以及以文搜文的 `HyDERetriever`。


In [ ]:
"""
Day 32 - 步骤一：检索前优化三板斧（Query Rewriting / Multi-Query / HyDE）
对应生产包：langchain.retrievers.multi_query / HypotheticalDocumentEmbedder
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import Dict, List, Tuple

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document
from src.day29_vector_store import StarMindVectorStore


class QueryRewriter:
    """
    企业级查询改写器：
    将员工口语、地域别称与公司黑话规范化为正式制度检索词（生产可叠加轻量 7B LLM）。
    """

    TERM_MAPPING: Dict[str, str] = {
        "魔都": "上海 一线城市",
        "帝都": "北京 一线城市",
        "鹏城": "深圳 一线城市",
        "住店": "酒店住宿标准上限",
        "吃饭": "餐饮补助发票",
        "推线上": "生产发布窗口上线变更",
        "发版": "生产发布窗口代码合并",
        "礼拜五": "周五",
        "居家": "远程办公",
        "连不上网": "零信任VPN动态令牌",
        "请假条": "医院诊断证明病假",
        "年休": "带薪年假天数申请",
    }

    def rewrite(self, query: str) -> str:
        rewritten = query
        additions: List[str] = []
        for slang, formal in self.TERM_MAPPING.items():
            if slang in rewritten:
                additions.append(formal)
        if not additions:
            return query
        return f"{query} {' '.join(additions)}"


class MultiQueryRetriever:
    """
    多路查询检索器：
    将一个复杂或口语提问扩展为多个互补视角子查询，分别检索向量库后按内容去重取最高分合并。
    """

    def __init__(self, vector_store: StarMindVectorStore, rewriter: QueryRewriter | None = None) -> None:
        self.vector_store = vector_store
        self.rewriter = rewriter or QueryRewriter()

    def generate_queries(self, original_query: str) -> List[str]:
        queries: List[str] = [original_query]
        formal = self.rewriter.rewrite(original_query)
        if formal != original_query:
            queries.append(formal)

        # 拆解复合连接词（「和/以及/还有/顺便」）生成独立子查询
        for splitter in ("以及", "还有", "顺便", "和", "跟", "，", ","):
            if splitter in original_query:
                parts = [p.strip() for p in original_query.split(splitter) if len(p.strip()) >= 3]
                for p in parts:
                    rewritten_p = self.rewriter.rewrite(p)
                    if rewritten_p not in queries:
                        queries.append(rewritten_p)
                break
        return queries[:4]

    def retrieve(self, query: str, k: int = 3) -> List[Tuple[Document, float]]:
        sub_queries = self.generate_queries(query)
        merged: Dict[str, Tuple[Document, float]] = {}

        for sq in sub_queries:
            for doc, score in self.vector_store.similarity_search_with_score(sq, k=k):
                key = doc.page_content
                if key not in merged or score > merged[key][1]:
                    merged[key] = (doc, score)

        ranked = sorted(merged.values(), key=lambda x: x[1], reverse=True)
        return ranked[:k]


class HyDERetriever:
    """
    HyDE（Hypothetical Document Embeddings）假设性文档嵌入检索器：
    先生成一段符合企业制度文风的假设性回答，再以假设文档的向量检索真实知识库。
    """

    def __init__(self, vector_store: StarMindVectorStore) -> None:
        self.vector_store = vector_store

    def generate_hypothetical_doc(self, query: str) -> str:
        """离线模拟 LLM 生成制度风格的假设性段落（生产环境调用轻量 ChatModel）。"""
        if any(w in query for w in ("住店", "魔都", "帝都", "酒店", "出差", "吃饭", "报销")):
            return (
                "根据杭州星辰数科财务报销制度，员工因公前往一线城市（北京、上海、深圳、广州）或新一线城市出差，"
                "需遵守酒店住宿每晚标准上限与餐饮补助定额发放规定，并提前在OA完成审批。"
            )
        if any(w in query for w in ("礼拜五", "周五", "推线上", "发版", "合并", "代码")):
            return (
                "根据研发工程规范，业务代码合并至main分支须通过GitLab Merge Request评审与CI测试，"
                "日常生产发布窗口限定为周一至周四，严禁周五17:00之后非紧急变更上线。"
            )
        if any(w in query for w in ("居家", "内网", "VPN", "脱敏", "手机号")):
            return (
                "根据IT信息安全指南，员工远程办公接入内网须使用星辰零信任VPN与动态令牌，"
                "同时严格遵守客户敏感数据脱敏红线。"
            )
        if any(w in query for w in ("年休", "年假", "病假", "请假", "休假")):
            return (
                "根据HR员工休假手册，员工按工龄享有带薪年假并需提前提交OA申请，"
                "病假超过规定天数须提供公立医院诊断证明。"
            )
        return f"关于「{query}」的企业正式管理制度条款与执行标准说明。"

    def retrieve(self, query: str, k: int = 3) -> List[Tuple[Document, float]]:
        hypo_doc = self.generate_hypothetical_doc(query)
        return self.vector_store.similarity_search_with_score(hypo_doc, k=k)


if __name__ == "__main__":
    from src.day30_kb_cli import build_starmind_kb

    kb = build_starmind_kb()
    q = "去魔都出差住店和吃饭分别怎么报？"
    mq = MultiQueryRetriever(kb.vector_store)
    hyde = HyDERetriever(kb.vector_store)

    print("Multi-Query 展开:", mq.generate_queries(q))
    print("Multi-Query 命中:", [(d.metadata["section"], s) for d, s in mq.retrieve(q, k=2)])
    print("HyDE 命中:", [(d.metadata["section"], s) for d, s in hyde.retrieve(q, k=2)])


### 🔍 步骤 1 代码深度解析

1. `MultiQueryRetriever` 不仅做了术语改写，还把复合提问（「住店和吃饭」）拆成两条独立子查询，保证住宿标准和餐饮补助两个不同章节都能同时被召回！
2. `HyDERetriever` 把 10 字口语提问转化成一段制度风陈述句再去搜向量库，实现了经典的「Doc-to-Doc 向量空间对齐」。

**预期运行输出参考：**
```text
Multi-Query 展开: ['去魔都出差住店和吃饭分别怎么报？', '去魔都出差住店和吃饭分别怎么报？ 上海 一线城市 酒店住宿标准上限 餐饮补助发票', '去魔都出差住店 上海 一线城市 酒店住宿标准上限', '吃饭分别怎么报？ 餐饮补助发票']
Multi-Query 命中: [('差旅住宿标准', 0.6542), ('餐饮与交通补助', 0.5811)]
HyDE 命中: [('差旅住宿标准', 0.7125), ('餐饮与交通补助', 0.4219)]
```


## 💻 实操步骤 2：步骤二：实现上下文压缩器（ContextualCompressor）与召回率量化对比实验

- **目标文件**：`versions/day32/src/day32_compression_eval.py`
- **老陈代码旁白**：检索召回拉高了，接下来写一个句子级 `ContextualCompressor` 把无关背景句裁掉省 Token，并在一组刁钻的口语化/黑话测试集上跑通 Baseline vs QueryRewrite vs Multi-Query vs HyDE 召回率对比实验！


In [ ]:
"""
Day 32 - 步骤二：上下文压缩器（ContextualCompressor）与高级检索召回率对比实验
"""

from __future__ import annotations

import re
import sys
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document
from src.day29_embeddings_hnsw import StarMindLocalEmbeddings, cosine_similarity
from src.day30_kb_cli import build_starmind_kb
from src.day32_query_transform import HyDERetriever, MultiQueryRetriever, QueryRewriter


class ContextualCompressor:
    """
    句子级上下文压缩器：
    将召回的长切片按中文句末标点拆句，计算每句与查询的语义相似度，
    仅保留超过阈值的核心证据句，降低 LLM 输入 Token 与注意力噪音。
    """

    def __init__(
        self,
        embeddings: StarMindLocalEmbeddings | None = None,
        min_sentence_score: float = 0.12,
        max_sentences_per_doc: int = 2,
    ) -> None:
        self.embeddings = embeddings or StarMindLocalEmbeddings()
        self.min_sentence_score = min_sentence_score
        self.max_sentences_per_doc = max_sentences_per_doc

    def compress_documents(
        self,
        query: str,
        docs: Sequence[Document],
    ) -> Tuple[List[Document], float]:
        q_vec = self.embeddings.embed_query(query)
        orig_chars = sum(len(d.page_content) for d in docs)
        compressed: List[Document] = []

        for doc in docs:
            sentences = [s.strip() for s in re.split(r"(?<=[。！？；\n])", doc.page_content) if s.strip()]
            if not sentences:
                continue
            scored = [
                (cosine_similarity(q_vec, self.embeddings.embed_query(s)), idx, s)
                for idx, s in enumerate(sentences)
            ]
            # 筛选超过阈值的高相关句，至少保留最高分 1 句并还原原文顺序
            kept = [item for item in scored if item[0] >= self.min_sentence_score]
            if not kept:
                kept = [max(scored, key=lambda x: x[0])]
            kept_top = sorted(sorted(kept, key=lambda x: x[0], reverse=True)[: self.max_sentences_per_doc], key=lambda x: x[1])
            new_text = "".join(item[2] for item in kept_top)
            compressed.append(
                Document(
                    page_content=new_text,
                    metadata={**doc.metadata, "compressed": True, "orig_len": len(doc.page_content)},
                )
            )

        new_chars = sum(len(d.page_content) for d in compressed)
        saved_ratio = round(1.0 - (new_chars / max(1, orig_chars)), 4)
        return compressed, saved_ratio


# 专门构造贴近真实员工口语/黑话/复合意图的难题测试集
SLANG_EVAL_CASES = [
    ("去魔都住店最多能报多少？", "差旅住宿标准"),
    ("礼拜五下班前能不能推线上？", "生产发布窗口"),
    ("居家干活连不上网咋整？", "VPN与内网接入"),
    ("生病开请假条有啥要求？", "病假与证明"),
    ("老员工年休几天怎么申请？", "年假规定"),
    ("出差吃饭要贴票吗？", "餐饮与交通补助"),
]


def run_recall_comparison(top_k: int = 2) -> Dict[str, float]:
    kb = build_starmind_kb()
    vs = kb.vector_store
    rewriter = QueryRewriter()
    mq = MultiQueryRetriever(vs, rewriter)
    hyde = HyDERetriever(vs)

    strategies = {
        "Baseline_Naive": lambda q: vs.similarity_search_with_score(q, k=top_k),
        "QueryRewriting": lambda q: vs.similarity_search_with_score(rewriter.rewrite(q), k=top_k),
        "MultiQuery": lambda q: mq.retrieve(q, k=top_k),
        "HyDE": lambda q: hyde.retrieve(q, k=top_k),
    }

    report: Dict[str, float] = {}
    for name, fn in strategies.items():
        hits = 0
        for q, expected_section in SLANG_EVAL_CASES:
            docs = [d for d, _ in fn(q)]
            if any(d.metadata.get("section") == expected_section for d in docs):
                hits += 1
        report[name] = round(hits / len(SLANG_EVAL_CASES), 4)
    return report


if __name__ == "__main__":
    print("=== 口语/黑话难题集 Recall@2 对比实验 ===")
    for k_name, score in run_recall_comparison(top_k=2).items():
        print(f"  {k_name:16s} -> Recall@2 = {score:.2%}")

    # 演示上下文压缩节省 Token 效果
    long_doc = Document(
        "公司成立于2018年位于杭州未来科技城，拥有超过800名员工。"
        "一线城市（北京、上海、深圳、广州）酒店住宿上限每晚600元。"
        "园区地下车库免费向全体员工开放，请勿占用消防通道。"
        "食堂二楼提供杭帮菜和轻食沙拉。",
        {"source": "综合手册.md"},
    )
    comp = ContextualCompressor(max_sentences_per_doc=1)
    slim_docs, saved = comp.compress_documents("上海出差住酒店上限是多少", [long_doc])
    print(f"\n上下文压缩节省字符比例: {saved:.1%}")
    print("压缩后保留证据句:", slim_docs[0].page_content)


### 🔍 步骤 2 代码深度解析

1. `ContextualCompressor` 在挑出最相关句子后，特意按原句索引 `idx` 重新排回原文顺序（`key=lambda x: x[1]`），避免打乱前后逻辑因果关系。
2. 在 `SLANG_EVAL_CASES` 口语化难题集上，QueryRewriting、MultiQuery 和 HyDE 均能把 Recall@2 稳定拉升至 100%，直观证明检索前优化的实战价值。

**预期运行输出参考：**
```text
=== 口语/黑话难题集 Recall@2 对比实验 ===
  Baseline_Naive   -> Recall@2 = 66.67%
  QueryRewriting   -> Recall@2 = 100.00%
  MultiQuery       -> Recall@2 = 100.00%
  HyDE             -> Recall@2 = 100.00%

上下文压缩节省字符比例: 72.1%
压缩后保留证据句: 一线城市（北京、上海、深圳、广州）酒店住宿上限每晚600元。
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day32/tests/test_day32.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 32 自动化单元测试：验证 Query改写、MultiQuery、HyDE、上下文压缩与召回率提升。"""

import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document
from src.day30_kb_cli import build_starmind_kb
from src.day32_compression_eval import ContextualCompressor, run_recall_comparison
from src.day32_query_transform import HyDERetriever, MultiQueryRetriever, QueryRewriter


def test_query_rewriter_and_multiquery():
    rw = QueryRewriter()
    out = rw.rewrite("礼拜五能不能推线上？")
    assert "周五" in out and "生产发布窗口" in out

    kb = build_starmind_kb()
    mq = MultiQueryRetriever(kb.vector_store, rw)
    sub_qs = mq.generate_queries("去魔都出差住店和吃饭怎么报销？")
    assert len(sub_qs) >= 3
    hits = mq.retrieve("去魔都出差住店和吃饭怎么报销？", k=2)
    sections = {d.metadata["section"] for d, _ in hits}
    assert "差旅住宿标准" in sections


def test_hyde_and_recall_improvement():
    report = run_recall_comparison(top_k=2)
    assert report["QueryRewriting"] >= report["Baseline_Naive"]
    assert report["MultiQuery"] >= report["Baseline_Naive"]
    assert report["HyDE"] >= 0.80


def test_contextual_compressor_saves_tokens():
    doc = Document(
        "园区班车每天早8点从地铁站发车。一线城市住宿上限每晚600元。周五下午茶在3楼茶水间领取。",
        {"source": "guide.md"},
    )
    comp = ContextualCompressor(max_sentences_per_doc=1)
    slim, saved = comp.compress_documents("一线城市住宿上限", [doc])
    assert saved > 0.4
    assert "600元" in slim[0].page_content
    assert "下午茶" not in slim[0].page_content

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：作业一：实现 Step-Back Prompting（退步抽象查询）检索增强器

**题目要求：**
当员工问一个极其具体的案例（如「我从杭州坐高铁去北京住全季酒店花了650元能全报吗？」）时，先退一步抽象成高层次制度问题（「一线城市出差住宿报销上限标准是什么？」）再检索。请实现 `step_back_rewrite(query)` 函数并验证其对具体案例的检索命中率。

**思路提示：** 提取具体场景中的核心实体（城市/金额/行为），映射为对应的抽象制度问题，再与原问题合并检索。


In [ ]:
"""Day 32 作业一参考答案：Step-Back 退步抽象检索增强"""
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day30_kb_cli import build_starmind_kb


def step_back_rewrite(specific_query: str) -> str:
    if any(k in specific_query for k in ("全季", "如家", "酒店", "花", "元", "北京", "上海")):
        return "一线城市与新一线城市差旅酒店住宿报销上限标准是什么？"
    if any(k in specific_query for k in ("感冒", "发烧", "请2天", "医院")):
        return "员工申请病假需要提供什么证明及薪资发放标准？"
    return specific_query


if __name__ == "__main__":
    kb = build_starmind_kb()
    q = "我昨天去北京住全季花了650块超标了吗？"
    abstract_q = step_back_rewrite(q)
    hits = kb.vector_store.similarity_search_with_score(abstract_q, k=1)
    print("退步抽象查询:", abstract_q)
    print("命中章节:", hits[0][0].metadata["section"], "score=", hits[0][1])
    assert hits[0][0].metadata["section"] == "差旅住宿标准"


**答案解析与评分标准：**

Step-Back Prompting 是 Google DeepMind 提出的经典技巧，专治「用户举了一个超长具体案例，直接检索反而被无关细节稀释向量」的痛点。


### 作业 2（企业实战）：作业二：实现基于多轮历史的指代消解改写器（CoreferenceRewriter）

**题目要求：**
当上一轮聊的是「年假规定」，本轮用户只问了「它需要提前几天申请？」或「那病假呢？」，请实现 `rewrite_with_history(history_questions, current_query)`，自动将「它/这个/那…」替换为上一轮的主题词，生成可独立检索的完整问句。

**思路提示：** 检测 `current_query` 是否包含指代词（`它/这个/那个/那`）或长度过短，从最近一轮历史提取主题实体进行补全。


In [ ]:
"""Day 32 作业二参考答案：多轮对话指代消解查询改写器"""
from typing import List


def rewrite_with_history(history_questions: List[str], current_query: str) -> str:
    if not history_questions:
        return current_query
    last_q = history_questions[-1]
    pronouns = ("它", "这个", "那个", "该制度", "这项")
    if any(p in current_query for p in pronouns) or (current_query.startswith("那") and len(current_query) <= 8):
        # 提取上一轮的核心主题词作为上下文锚点
        topic = last_q.replace("请问", "").replace("？", "").replace("?", "")[:14]
        resolved = current_query
        for p in pronouns:
            resolved = resolved.replace(p, f"「{topic}」")
        if resolved == current_query:
            resolved = f"结合前文「{topic}」，{current_query}"
        return resolved
    return current_query


if __name__ == "__main__":
    hist = ["杭州星辰数科年假有几天？"]
    r1 = rewrite_with_history(hist, "它需要提前几天申请？")
    r2 = rewrite_with_history(hist, "那病假呢？")
    print("改写1:", r1)
    print("改写2:", r2)
    assert "年假" in r1 and "年假" in r2


**答案解析与评分标准：**

这是多轮 RAG 系统的必经关口（Day 37 项目二会正式接入 FastAPI 链路），解决了追问场景下检索失灵的核心痛点。


### 作业 3（架构进阶）：作业三：实现子问题拆解与子答案汇总器（QueryDecomposition）

**题目要求：**
当员工一次问两个跨部门问题（「入职满一年有几天年假，另外周五晚上能发布代码吗？」）时，实现 `decompose_and_answer(kb, complex_query)`：按标点/连接词拆成多个子问题，分别调用 `kb.ask()` 检索回答，最后汇总成带统一引用列表的完整回复。

**思路提示：** 用正则按 `[，,。；;]|另外|以及|还有` 拆分，过滤掉短于 4 字的碎片，分别调用 `kb.ask()` 后合并 `answer` 和 `citations`。


In [ ]:
"""Day 32 作业三参考答案：跨部门复合问题拆解与汇总器"""
import re
import sys
from pathlib import Path
from typing import Any, Dict, List

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day30_kb_cli import build_starmind_kb


def decompose_and_answer(kb, complex_query: str) -> Dict[str, Any]:
    parts = [p.strip() for p in re.split(r"[，,。；;]|另外|以及|还有", complex_query) if len(p.strip()) >= 4]
    sub_answers: List[str] = []
    all_sources: List[str] = []
    for idx, sub_q in enumerate(parts, start=1):
        resp = kb.ask(sub_q)
        sub_answers.append(f"【子问题{idx}：{sub_q}】\n{resp.answer}")
        for c in resp.citations:
            if c.source not in all_sources:
                all_sources.append(c.source)
    return {
        "sub_questions": parts,
        "combined_answer": "\n\n".join(sub_answers),
        "sources": all_sources,
    }


if __name__ == "__main__":
    kb = build_starmind_kb()
    kb.score_threshold = 0.12  # 短子问题场景适度放宽向量相似度阈值以保证跨部门召回
    out = decompose_and_answer(kb, "员工入职满一年有多少天年假，另外周五晚上能发布代码上线吗？")
    print("拆解子问题:", out["sub_questions"])
    print("涉及文档:", out["sources"])
    assert len(out["sources"]) >= 2


**答案解析与评分标准：**

跨部门复合提问是 Naive RAG 的天然软肋（单次 Top-3 容易全被一个部门占满）。Query Decomposition（问题拆解）完美解决了多跳/多意图问答。


---

## 🎯 老陈收尾复盘

今天这四招（Query Rewrite / Multi-Query / HyDE / 上下文压缩）是面试和企业落地的高频杀手锏。大家也从刚才的量化对比实验看到了：面对口语化/黑话提问，Multi-Query 和 HyDE 能把召回率从基线一路拉满，而上下文压缩直接帮我们省掉 50% 以上的冗余输入字符！明天咱们继续进阶，攻克 BM25 + 向量混合检索与 Reranker 重排。
